In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/__huggingface_repos__.json
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/__notebook__.ipynb
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/__output__.json
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/__results__.html
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/custom.css
/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/reviews_with_sentiment.csv


In [2]:
import pandas as pd, spacy
from collections import Counter

df = pd.read_csv ("/kaggle/input/notebooks/kylin0338/01-sentiment-pipeline/reviews_with_sentiment.csv")
df["review_id"] = df.index
nlp = spacy.load("en_core_web_sm", disable=["ner"])

In [3]:
sample = df["text"].sample(5000, random_state=1).tolist()
counter = Counter()
for doc in nlp.pipe(sample, batch_size=256):
    for chunk in doc.noun_chunks:
        counter[chunk.root.lemma_.lower()] += 1

print(counter.most_common(100))

[('i', 11853), ('it', 8457), ('phone', 7867), ('you', 2179), ('that', 1562), ('this', 1371), ('they', 1194), ('screen', 973), ('camera', 687), ('app', 675), ('battery', 662), ('issue', 578), ('which', 549), ('time', 547), ('price', 540), ('life', 518), ('problem', 492), ('what', 471), ('card', 459), ('one', 414), ('thing', 407), ('quality', 396), ('device', 382), ('day', 374), ('product', 369), ('we', 361), ('feature', 344), ('iphone', 336), ('case', 328), ('everything', 327), ('call', 324), ('charger', 301), ('month', 295), ('samsung', 289), ('review', 274), ('note', 272), ('he', 259), ('picture', 253), ('all', 253), ('money', 250), ('amazon', 246), ('service', 245), ('update', 241), ('condition', 239), ('use', 234), ('seller', 222), ('lot', 219), ('year', 211), ('box', 207), ('verizon', 194), ('work', 193), ('photo', 193), ('button', 192), ('speaker', 190), ('hour', 187), ('people', 183), ('version', 183), ('video', 181), ('scratch', 175), ('android', 172), ('way', 171), ('edge', 168

In [4]:
ASPECTS = {
    "battery": ["battery", "battery life", "charge", "charger", "charging", "drain", "power bank"],
    "camera": ["camera", "photo", "picture", "video", "selfie", "lens", "flash", "autofocus"],
    "screen": ["screen", "display", "touchscreen", "brightness", "resolution", "amoled", "lcd", "screen protector"],
    "price": ["price", "cost", "value for money", "worth", "affordable", "cheap", "expensive", "overpriced"],
    "performance": ["performance", "speed", "lag", "laggy", "freeze", "crash", "processor", "ram", "smooth", "multitask"],
    "sound": ["sound", "speaker", "volume", "audio", "ringtone", "earpiece", "headphone jack"],
    "build": ["build quality", "durable", "durability", "material", "fragile", "sturdy", "scratch", "cracked", "plastic", "metal body"],
    "design": ["design", "size", "weight", "lightweight", "color", "slim", "thin", "sleek", "looks", "bezel"],
    "usability": ["easy to use", "user friendly", "interface", "intuitive", "fingerprint", "face unlock", "navigation"],
    "software": ["software", "app", "update", "android", "ios", "bloatware", "operating system", "firmware"],
    "network": ["network", "signal", "reception", "carrier", "verizon", "sim", "lte", "4g", "5g", "wifi", "dropped calls", "coverage"],
    "storage": ["storage", "sd card", "microsd", "memory", "internal storage"],
    "delivery": ["shipping", "delivery", "delivered", "arrived", "package", "packaging", "box"],
    "customer_service": ["seller", "customer service", "support", "refund", "return", "warranty", "replacement"],
}

In [5]:
import re

PATTERNS = {
    asp: re.compile(
        r"\b(?:" + "|".join(re.escape(w) for w in words) + r")s?\b"
    )
    for asp, words in ASPECTS.items()
}

def find_aspects(sentence):
    s = sentence.lower()
    return [
        asp for asp, pat in PATTERNS.items()
        if pat.search(s)
    ]
texts = df["text"].tolist()
ids = df["review_id"].tolist()
rows = []
for rid, doc in zip(ids, nlp.pipe(texts, batch_size=256)):
    for sent in doc.sents:
        found = find_aspects(sent.text)
        if 1 <= len(found) <= 2 and len(sent.text.split()) >= 4:
            for asp in found:
                rows.append({"review_id": rid, "sentence": sent.text.strip(), "aspect": asp})

In [6]:
pairs = pd.DataFrame(rows)

print("Aspect distribution:")
print(pairs["aspect"].value_counts(normalize=True).round(3))

print("\nSample rows:")
print(pairs.sample(min(10, len(pairs)), random_state=42).to_string(index=False))
print(
    pairs.sample(min(10, len(pairs)), random_state=42)
         .to_string(index=False)
)
# print(pd.crosstab(pairs["aspect"], pairs["aspect_sentiment"], normalize="index").round(2))

Aspect distribution:
aspect
battery             0.144
software            0.128
network             0.113
screen              0.108
camera              0.104
price               0.077
customer_service    0.075
design              0.056
delivery            0.044
sound               0.039
performance         0.036
usability           0.029
storage             0.024
build               0.023
Name: proportion, dtype: float64

Sample rows:
 review_id                                                                                                                                 sentence    aspect
      3423                                                                                    Its interface is easy on those of us with older eyes. usability
      8819                                                     The software on the phone itself is very good and there is no bloatware to be found.  software
     13503                                                                             

In [7]:
pairs = pd.DataFrame(rows)
print(pairs.shape)
print(pairs["aspect"].value_counts(normalize=True).round(3))
# print(pd.crosstab(pairs["aspect"], pairs["aspect_sentiment"], normalize="index").round(2))

(46483, 3)
aspect
battery             0.144
software            0.128
network             0.113
screen              0.108
camera              0.104
price               0.077
customer_service    0.075
design              0.056
delivery            0.044
sound               0.039
performance         0.036
usability           0.029
storage             0.024
build               0.023
Name: proportion, dtype: float64


In [8]:
from transformers import pipeline

absa = pipeline(
    "text-classification",
    model="yangheng/deberta-v3-base-absa-v1.1",
    device=0, truncation=True, max_length=256
)

inputs = [{"text": s, "text_pair": a} for s, a in zip(pairs["sentence"], pairs["aspect"])]
out = absa(inputs, batch_size=64)

pairs["aspect_sentiment"] = [o["label"].lower() for o in out]
pairs["aspect_score"] = [o["score"] for o in out]

# !pip install -q sentencepiece
# from transformers import pipeline

# absa = pipeline(
#     "text-classification",
#     model="yangheng/deberta-v3-base-absa-v1.1",
#     device=0, truncation=True, max_length=256
# )

# inputs = [{"text": s, "text_pair": a} for s, a in zip(pairs["sentence"], pairs["aspect"])]
# out = absa(inputs, batch_size=64)

# pairs["aspect_sentiment"] = [o["label"].lower() for o in out]
# pairs["aspect_score"] = [o["score"] for o in out]

config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

In [9]:
print(pd.crosstab(pairs["aspect"], pairs["aspect_sentiment"], normalize="index").round(2))

aspect_sentiment  negative  neutral  positive
aspect                                       
battery               0.40     0.25      0.35
build                 0.36     0.14      0.50
camera                0.26     0.29      0.45
customer_service      0.55     0.27      0.18
delivery              0.29     0.17      0.53
design                0.20     0.08      0.72
network               0.45     0.35      0.21
performance           0.30     0.08      0.61
price                 0.26     0.10      0.64
screen                0.42     0.21      0.37
software              0.36     0.39      0.26
sound                 0.50     0.13      0.36
storage               0.35     0.34      0.31
usability             0.33     0.08      0.58


In [11]:
pairs = pairs.merge(df[["review_id", "asin", "rating"]], on="review_id")
pairs.to_csv("/kaggle/working/aspect_sentiment.csv", index=False)

In [12]:
for asp in ["design", "usability", "performance", "sound"]:
    # print(f"\n=== {asp} ===")
    for _, r in pairs[pairs["aspect"] == asp].sample(12, random_state=3).iterrows():
        print(f"[{r['aspect_sentiment']}] {r['sentence'][:150]}")

[positive] Amazing just git looks brand new came with a real apple charger everything works great came a day early amazing definitely buy
[positive] Plus the design is great and very nicely sticks out in a world full of iPhones and Galaxies.
[negative] I'll update the review if Samsung can show it was a fluke and not just poor design.
[positive] Screen size is good for my hand and fits nicely in my pocket.
[positive] The phone itself is about the size of a Hershey bar.
[positive] In fact its screen size is spec’d at exactly the same 6” size as a Kindle but in a device that’s a fraction of the size.
[positive] So far it's working great and looks like a brand new phone.
[positive] This is my first full size smart phone and the ease of use is amazing.
[positive] However, given weight and beautiful/slippery finish I've found a case is necessary (don't like cases on flagship devices).
[neutral] You can change text size, set individual ringtones and notification sounds, change the shortcuts 

In [13]:
val = pairs.groupby("aspect").sample(20, random_state=1)[
    ["review_id", "sentence", "aspect", "aspect_sentiment", "aspect_score"]
]
val["aspect_correct"] = ""      # 1 if sentence is really about that aspect, else 0
val["sentiment_correct"] = ""   # 1 if the sentiment label is right, else 0
val.to_csv("/kaggle/working/validation.csv", index=False)

In [ ]:
# v = pd.read_csv("/kaggle/input//validation.csv")
# print(v.groupby("aspect")[["aspect_correct", "sentiment_correct"]].mean().round(2))
# print(v[["aspect_correct", "sentiment_correct"]].mean().round(3))

In [ ]:
# from sklearn.metrics import classification_report

# # test_df needs: sentence, aspect_term, label (map labels to negative/neutral/positive)
# inputs = [{"text": s, "text_pair": a} for s, a in zip(test_df["sentence"], test_df["aspect_term"])]
# preds = [o["label"].lower() for o in absa(inputs, batch_size=64)]
# print(classification_report(test_df["label"], preds))